# 00 — Verificación del entorno

Este notebook comprueba que todas las dependencias del TFM están instaladas y funcionan. Si todas las celdas se ejecutan sin error, el entorno está listo para empezar.

Si alguna celda de red falla, anota el error y continua con los datos de ejemplo. Si falla la importacion de paquetes, revisa `environment.yml`.

## 1. Versiones de librerías core

In [1]:
import sys
import numpy as np
import pandas as pd
import scipy
import Bio
import matplotlib
import seaborn

print(f'Python    : {sys.version.split()[0]}')
print(f'NumPy     : {np.__version__}')
print(f'pandas    : {pd.__version__}')
print(f'SciPy     : {scipy.__version__}')
print(f'Biopython : {Bio.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'seaborn   : {seaborn.__version__}')

Python    : 3.11.15
NumPy     : 2.4.6
pandas    : 3.0.3
SciPy     : 1.17.1
Biopython : 1.87
matplotlib: 3.10.9
seaborn   : 0.13.2


## 2. Configuración del proyecto

In [2]:
from tfm_ras.config import load_config, project_root

root = project_root()
print(f'Raíz del proyecto: {root}')

cfg = load_config(root / 'configs' / 'config.yaml')
print(f'Familia: {cfg["family"]["name"]}')
for isoform, m in cfg['family']['members'].items():
    print(f'  - {isoform}: '
          f'UniProt {m["uniprot_id"]}, '
          f'PDB {m["pdbs"]["primary"]["id"]}'
    )

Raíz del proyecto: /Users/rachi/Desktop/TFM/tfm_ras_mutations
Familia: RAS
  - KRAS: UniProt P01116, PDB 4OBE
  - HRAS: UniProt P01112, PDB 3K8Y
  - NRAS: UniProt P01111, PDB 5UHV


## 3. Conexión a UniProt y RCSB PDB

In [3]:
import requests

r = requests.get('https://rest.uniprot.org/uniprotkb/P01116.fasta', timeout=10)
assert r.status_code == 200, 'No se puede conectar con UniProt'
print('UniProt OK')
print(r.text.split(chr(10))[0])

r = requests.get('https://files.rcsb.org/download/4OBE.pdb', timeout=10)
assert r.status_code == 200, 'No se puede conectar con RCSB'
print('RCSB PDB OK')
print(f'Tamaño 4OBE: {len(r.text)} caracteres')

UniProt OK
>sp|P01116|RASK_HUMAN GTPase KRas OS=Homo sapiens OX=9606 GN=KRAS PE=1 SV=1
RCSB PDB OK
Tamaño 4OBE: 750465 caracteres


## 4. Visualización 3D (py3Dmol)

In [4]:
import py3Dmol

view = py3Dmol.view(query='pdb:4OBE', width=400, height=300)
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view  # debe mostrar la estructura interactiva

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 5. Tests unitarios

In [5]:
# Solo verifica que pytest descubre los tests correctamente.
# Los tests iniciales deben pasar; el alumno debe ampliarlos en los hitos 1 y 2.
import subprocess
result = subprocess.run(['pytest', '--collect-only', '-q'], capture_output=True, text=True, cwd=str(root))
print(result.stdout)
print('STDERR:', result.stderr[:500] if result.stderr else '(vacío)')

tests/test_basic.py::test_parse_aa_change_g12d
tests/test_basic.py::test_parse_aa_change_q61r
tests/test_basic.py::test_parse_aa_change_rejects_invalid_format
tests/test_basic.py::test_expected_cosmic_columns_are_documented
tests/test_basic.py::test_missing_expected_columns_reports_absent_columns
tests/test_basic.py::test_curated_schema_contains_minimum_columns

6 tests collected in 0.95s

STDERR: (vacío)


## ✅ Si todas las celdas se ejecutan, ya puedes empezar con `01_data_curation.ipynb`.